# LeetCode #1462: Course Schedule IV

https://leetcode.com/problems/course-schedule-iv/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (BFS per query)** | $O(q \cdot (n + m))$ | $O(n + m)$ |
| **Optimal: Floyd-Warshall Reachability ★** | $O(n^3 + q)$ | $O(n^2)$ |

---

## Understanding the Methods

### Brute Force (BFS per query)
For each query $(u, v)$, run a BFS/DFS from $u$ to see if $v$ is reachable. With $q$ up to $n^2$ this can be $O(n^3)$ in the worst case with extra overhead.

### Optimal: Floyd-Warshall Reachability ★
Build a boolean reachability matrix via Floyd-Warshall: $reach[i][j] = true$ if $j$ is reachable from $i$ through any path. Each query is then answered in $O(1)$. Total: $O(n^3 + m + q)$.

**Constraints:**
* $2 \le n \le 100$
* $0 \le prerequisites.length \le n(n-1)/2$
* $0 \le queries.length \le 10^4$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;

public class Solution {
    public IList<bool> CheckIfPrerequisite(int numCourses, int[][] prerequisites, int[][] queries) {
        int n = numCourses;
        // reach[i][j] = true means course i is a prerequisite (direct or indirect) of course j
        bool[,] reach = new bool[n, n];

        // Seed direct prerequisites
        foreach (var p in prerequisites) reach[p[0], p[1]] = true;

        // Floyd-Warshall: propagate reachability through intermediate nodes
        for (int k = 0; k < n; k++)
            for (int i = 0; i < n; i++)
                for (int j = 0; j < n; j++)
                    if (reach[i, k] && reach[k, j]) reach[i, j] = true;

        // Answer each query in O(1)
        var result = new List<bool>();
        foreach (var q in queries) result.Add(reach[q[0], q[1]]);
        return result;
    }
}

### Python

In [ ]:
class Solution:
    def checkIfPrerequisite(self, numCourses: int, prerequisites: list[list[int]], queries: list[list[int]]) -> list[bool]:
        n = numCourses
        # reach[i][j] means i must be taken before j (directly or transitively)
        reach = [[False] * n for _ in range(n)]
        for u, v in prerequisites:
            reach[u][v] = True

        # Propagate transitive reachability
        for k in range(n):
            for i in range(n):
                for j in range(n):
                    if reach[i][k] and reach[k][j]:
                        reach[i][j] = True

        return [reach[u][v] for u, v in queries]

### Go

In [ ]:
func checkIfPrerequisite(numCourses int, prerequisites [][]int, queries [][]int) []bool {
	n := numCourses
	// reach[i][j]: i is an indirect or direct prerequisite of j
	reach := make([][]bool, n)
	for i := range reach { reach[i] = make([]bool, n) }
	for _, p := range prerequisites { reach[p[0]][p[1]] = true }

	// Floyd-Warshall to compute transitive closure
	for k := 0; k < n; k++ {
		for i := 0; i < n; i++ {
			for j := 0; j < n; j++ {
				if reach[i][k] && reach[k][j] { reach[i][j] = true }
			}
		}
	}

	result := make([]bool, len(queries))
	for idx, q := range queries { result[idx] = reach[q[0]][q[1]] }
	return result
}

### Rust

In [ ]:
impl Solution {
    pub fn check_if_prerequisite(num_courses: i32, prerequisites: Vec<Vec<i32>>, queries: Vec<Vec<i32>>) -> Vec<bool> {
        let n = num_courses as usize;
        let mut reach = vec![vec![false; n]; n];
        for p in &prerequisites { reach[p[0] as usize][p[1] as usize] = true; }

        // Floyd-Warshall transitive closure
        for k in 0..n {
            for i in 0..n {
                for j in 0..n {
                    if reach[i][k] && reach[k][j] { reach[i][j] = true; }
                }
            }
        }

        queries.iter().map(|q| reach[q[0] as usize][q[1] as usize]).collect()
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `numCourses=2, prerequisites=[[1,0]], queries=[[0,1],[1,0]]`
Course 1 must precede 0. Query [0,1]: is 0 a prerequisite of 1? No. Query [1,0]: is 1 a prerequisite of 0? Yes. Answer: **[false, true]**.

### 2. Slightly Complex
**Input:** `numCourses=3, prerequisites=[[1,2],[1,0],[2,0]], queries=[[1,0],[1,2]]`
1→2→0 and 1→0 directly. Both queries are true: **[true, true]**.

### 3. Edge Case: Time Factor
**Input:** $n=100$, prerequisites forming a chain $0 \to 1 \to \cdots \to 99$, 10 000 queries.
Floyd-Warshall runs $100^3 = 10^6$ iterations once. Then 10 000 queries each answered in $O(1)$. Total: $O(10^6 + 10^4)$.

### 4. Edge Case: Space Factor
**Input:** $n=100$, any graph.
Reachability matrix is $100 \times 100 = 10^4$ booleans — $O(n^2)$, easily fits in memory.

### 5. Almost-Impossible but Plausible
**Input:** `numCourses=100`, prerequisites form a cycle (which the problem guarantees won't happen). As a sanity check: Floyd-Warshall would set $reach[i][i]=true$ for cyclic nodes, but since problem guarantees a DAG, self-loops never appear and all answers are well-defined.